# SI699: Data Processing Pipeline

This notebook loads the Reddit mental health corpus from Hugging Face, cleans and filters posts, and generates a sample for risk annotation.

**Steps:**
1. Install dependencies
2. Load corpus from Hugging Face
3. Build document text (title + body)
4. Clean text (URLs, HTML, whitespace)
5. Filter deleted/removed and short posts
6. Stratified sampling for annotation
7. Rule-based pre-annotation (Zirikly-style)

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Optional

print("Libraries imported successfully!")

# Paths - Use local relative paths
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face dataset
HF_DATASET_ID = "solomonk/reddit_mental_health_posts"
HF_SPLIT = "train"

# Subreddits in corpus
SUBREDDITS = ["ADHD", "Aspergers", "Depression", "OCD", "PTSD"]

# Risk levels (annotation scheme from Zirikly et al. CLPsych 2019)
RISK_LEVELS = {
    0: "No Risk",      # general stress/venting; no self-harm ideation
    1: "Low Risk",     # distress/hopelessness; no explicit self-harm wording
    2: "Moderate Risk", # self-harm/suicidal ideation; no clear plan
    3: "Severe Risk",   # ideation + plan/means/intent
}
HIGH_RISK_LABELS = {2, 3}

# Annotation subset settings
ANNOTATION_TARGET_SIZE = 1200
HIGH_RISK_KEYWORDS = [
    "suicide", "suicidal", "kill myself", "end it", "self-harm",
    "self harm", "want to die", "don't want to live", "end my life",
]

# Text preprocessing
MIN_POST_TOKENS = 10
MAX_BODY_LENGTH = 50000

# Reproducibility
RANDOM_STATE = 42

print("Configuration loaded!")
print(f"Data directory: {DATA_DIR}")

Libraries imported successfully!
Configuration loaded!
Data directory: data


## 3. Data Loading Functions

In [2]:
from datasets import load_dataset


def load_or_download_csv(csv_path: str = "data/reddit_mental_health_posts.csv"):
    """Load CSV from local path, or download from Hugging Face if not found."""
    csv_path_obj = Path(csv_path)
    
    if csv_path_obj.exists():
        print(f"Loading from local CSV: {csv_path}")
        return pd.read_csv(csv_path)
    else:
        print(f"Downloading dataset from Hugging Face...")
        dataset = load_dataset('solomonk/reddit_mental_health_posts')
        data = dataset['train'].to_pandas()
        
        # Save to local CSV for future use
        csv_path_obj.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(csv_path, index=False)
        print(f"Saved to: {csv_path}")
        return data

## 4. Text Preprocessing Functions

In [3]:
def clean_post_text(text: str) -> str:
    """
    Clean single post text: URLs, HTML, repeated whitespace; keep negations and emotion words.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text, flags=re.IGNORECASE)
    # HTML artifacts
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    text = re.sub(r"<[^>]+>", " ", text)
    # Reddit markdown links [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def filter_posts(
    df: pd.DataFrame,
    text_col: str = "text",
    min_tokens: int = 10,
    drop_deleted: bool = True,
) -> pd.DataFrame:
    """
    Filter out deleted/removed and extremely short posts.
    """
    out = df.copy()
    if drop_deleted:
        deleted_vals = ["[deleted]", "[removed]"]
        for col in ["body", "title"]:
            if col in out.columns:
                out = out[~out[col].fillna("").str.strip().str.lower().isin(deleted_vals)]
    if text_col in out.columns:
        token_count = out[text_col].fillna("").str.split().str.len()
        out = out[token_count >= min_tokens]
    return out.reset_index(drop=True)


def build_document_text(
    df: pd.DataFrame,
    title_col: str = "title",
    body_col: str = "body",
    output_col: str = "text",
    max_body_len: Optional[int] = None,
) -> pd.DataFrame:
    """
    Build single 'text' column from title + body for topic modeling.
    """
    out = df.copy()
    title = out[title_col].fillna("")
    body = out[body_col].fillna("")
    if max_body_len:
        body = body.str.slice(0, max_body_len)
    out[output_col] = (title + " " + body).str.strip()
    return out

## 5. Sampling Functions

In [4]:
def _has_risk_keyword(text: str, keywords: List[str]) -> bool:
    if not isinstance(text, str):
        return False
    lower = text.lower()
    return any(kw.lower() in lower for kw in keywords)


def sample_for_annotation(
    df: pd.DataFrame,
    text_col: str = "text",
    subreddit_col: str = "subreddit",
    target_size: int = ANNOTATION_TARGET_SIZE,
    high_risk_keywords: Optional[List[str]] = None,
    oversample_risk_ratio: float = 0.35,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Stratify by subreddit; oversample posts containing high-risk keywords; fill rest randomly.
    """
    if high_risk_keywords is None:
        high_risk_keywords = HIGH_RISK_KEYWORDS

    # Build text for keyword check
    if text_col not in df.columns and "title" in df.columns and "body" in df.columns:
        text = df["title"].fillna("") + " " + df["body"].fillna("")
    else:
        text = df[text_col] if text_col in df.columns else df.get("body", pd.Series([""] * len(df)))

    risk_mask = text.apply(lambda t: _has_risk_keyword(t, high_risk_keywords))
    n_risk = risk_mask.sum()
    n_oversample = min(int(target_size * oversample_risk_ratio), n_risk)

    rng = np.random.default_rng(random_state)
    out_rows = []

    # 1) Oversample high-risk keyword posts
    risk_idx = df.index[risk_mask].tolist()
    if risk_idx:
        n_take = min(n_oversample, len(risk_idx))
        chosen = rng.choice(risk_idx, size=n_take, replace=False)
        out_rows.append(df.loc[chosen])
    n_taken = sum(len(r) for r in out_rows)
    remaining = target_size - n_taken

    # 2) Stratified sample from rest (by subreddit)
    other_mask = ~df.index.isin(out_rows[0].index) if out_rows else pd.Series(True, index=df.index)
    other = df.loc[other_mask]
    if other.empty or remaining <= 0:
        return pd.concat(out_rows, ignore_index=True) if out_rows else df.head(0)

    if subreddit_col in other.columns:
        strat = other.groupby(subreddit_col, group_keys=False).apply(
            lambda g: g.sample(n=min(len(g), max(1, remaining // other[subreddit_col].nunique())), random_state=int(rng.integers(1e9))),
            include_groups=False,
        )
        strat = strat.sample(n=min(remaining, len(strat)), random_state=int(rng.integers(1e9)))
    else:
        strat = other.sample(n=min(remaining, len(other)), random_state=int(rng.integers(1e9)))

    out_rows.append(strat)
    return pd.concat(out_rows, ignore_index=True).drop_duplicates(subset=["id"] if "id" in df.columns else None)

## 6. Load and Process Corpus

In [5]:
print("=" * 60)
print("1. Loading corpus from local or Hugging Face...")
print("=" * 60)

df = load_or_download_csv(csv_path=str(DATA_DIR / "reddit_mental_health_posts.csv"))
print(f"   Raw count: {len(df):,}")

if "subreddit" in df.columns:
    print("   Per-subreddit counts:")
    for sr, cnt in df["subreddit"].value_counts().items():
        print(f"      {sr}: {cnt:,}")

if "created_utc" in df.columns:
    df["created_utc"] = df["created_utc"].astype(str)
    try:
        t = pd.to_datetime(df["created_utc"], errors="coerce")
        print(f"   Time range: {t.min()} ~ {t.max()}")
    except Exception:
        pass

1. Loading corpus from local or Hugging Face...
Loading from local CSV: data/reddit_mental_health_posts.csv
   Raw count: 151,288
   Per-subreddit counts:
      OCD: 42,826
      ADHD: 37,109
      depression: 24,031
      ptsd: 24,028
      aspergers: 23,294
   Time range: 2019-07-07 16:21:24+00:00 ~ 2021-12-23 18:01:41+00:00


In [6]:
print("\n" + "=" * 60)
print("2. Building document text (title + body)...")
print("=" * 60)

df = build_document_text(df, max_body_len=MAX_BODY_LENGTH)
df["body"] = df["body"].fillna("").astype(str)
df["title"] = df["title"].fillna("").astype(str)
df["text"] = (df["title"] + " " + df["body"]).str.strip()

print(f"   Done. Sample text length: {df['text'].str.len().mean():.0f} chars (avg)")


2. Building document text (title + body)...
   Done. Sample text length: 605 chars (avg)


In [7]:
print("\n" + "=" * 60)
print("3. Cleaning text (URL/HTML/whitespace)...")
print("=" * 60)

df["text"] = df["text"].apply(clean_post_text)
print(f"   Done. Sample cleaned text length: {df['text'].str.len().mean():.0f} chars (avg)")


3. Cleaning text (URL/HTML/whitespace)...
   Done. Sample cleaned text length: 600 chars (avg)


In [8]:
print("\n" + "=" * 60)
print("4. Filtering [deleted]/[removed] and too-short posts...")
print("=" * 60)

before = len(df)
deleted_vals = ["[deleted]", "[removed]"]
body_deleted = df["body"].fillna("").str.strip().str.lower().isin(deleted_vals)
title_deleted = df["title"].fillna("").str.strip().str.lower().isin(deleted_vals)
n_deleted = (body_deleted | title_deleted).sum()
df_after_del = df[~body_deleted & ~title_deleted]
token_count = df_after_del["text"].fillna("").str.split().str.len()
n_too_short = (token_count < MIN_POST_TOKENS).sum()
df = filter_posts(df, text_col="text", min_tokens=MIN_POST_TOKENS)

print(f"   Before filter: {before:,} posts")
print(f"   - body/title [deleted]/[removed]: removed {n_deleted:,}")
print(f"   - remaining with < {MIN_POST_TOKENS} tokens: removed {n_too_short:,}")
print(f"   After filter: {len(df):,} posts (total removed: {before - len(df):,})")


4. Filtering [deleted]/[removed] and too-short posts...
   Before filter: 151,288 posts
   - body/title [deleted]/[removed]: removed 62,601
   - remaining with < 10 tokens: removed 954
   After filter: 87,733 posts (total removed: 63,555)


In [9]:
# Save processed corpus
out_corpus = DATA_DIR / "processed_corpus.csv"
df.to_csv(out_corpus, index=False)
print(f"\n   Saved: {out_corpus}")


   Saved: data/processed_corpus.csv


## 7. Sampling for Annotation

In [10]:
print("\n" + "=" * 60)
print("5. Sampling for annotation (stratified + high-risk keyword oversample)...")
print("=" * 60)

sampled = sample_for_annotation(
    df,
    text_col="text",
    subreddit_col="subreddit",
    target_size=ANNOTATION_TARGET_SIZE,
)
sample_path = DATA_DIR / "sample_for_annotation.csv"
sampled.to_csv(sample_path, index=False)
print(f"   Sample size: {len(sampled):,} -> {sample_path}")

if "subreddit" in sampled.columns:
    print("   Per-subreddit in sample:")
    for sr, cnt in sampled["subreddit"].value_counts().items():
        print(f"      {sr}: {cnt}")


5. Sampling for annotation (stratified + high-risk keyword oversample)...
   Sample size: 1,200 -> data/sample_for_annotation.csv
   Per-subreddit in sample:
      depression: 206
      OCD: 95
      ptsd: 79
      aspergers: 22
      ADHD: 18


## 8. Rule-Based Pre-Annotation (Zirikly-style)

In [11]:
# --- Rule definitions (Zirikly-style) ---

# Severe (3): plan / means / intent
SEVERE_PLAN = re.compile(
    r"\b(plan\s+to|planning\s+to|going\s+to\s+(kill|end\s+it|do\s+it)|will\s+kill\s+myself|"
    r"tonight\s+I'll|tomorrow\s+I'll|this\s+(week|weekend)|have\s+a\s+plan|my\s+plan\s+is|"
    r"when\s+I\s+(kill|do\s+it)|how\s+I'll\s+(kill|end)|ready\s+to\s+(die|end))\b",
    re.I,
)
SEVERE_MEANS = re.compile(
    r"\b(gun|pills|overdose|hang\s+(my|myself)|jump\s+off|bridge|rope|blade|"
    r"slash\s+(my|wrists)|cut\s+my\s+(wrists|throat)|shoot\s+myself|"
    r"carbon\s+monoxide|bleach|poison)\b",
    re.I,
)
SEVERE_INTENT = re.compile(
    r"\b(going\s+to\s+do\s+it|gonna\s+do\s+it|done\s+with\s+(life|everything)|"
    r"finally\s+(end|do\s+it)|end\s+it\s+(tonight|today|tomorrow)|"
    r"light\s+myself\s+on\s+fire|shoot\s+myself\s+in\s+the\s+head)\b",
    re.I,
)

# Moderate (2): clear ideation, no plan
MODERATE_IDEATION = re.compile(
    r"\b(kill\s+myself|want\s+to\s+die|suicidal|want\s+to\s+end\s+my\s+life|"
    r"don'?t\s+want\s+to\s+live|self[- ]harm|cut\s+myself|hurt\s+myself|"
    r"thoughts?\s+of\s+suicide|thinking\s+about\s+suicide|"
    r"end\s+it\s+all|want\s+to\s+disappear|end\s+my\s+life)\b",
    re.I,
)
MODERATE_SELFHARM = re.compile(
    r"\b(cutting\s+myself|cut\s+myself|self[- ]harm|self\s*harm|"
    r"intrusive\s+thoughts?\s+to\s+kill|thoughts?\s+of\s+killing)\b",
    re.I,
)

# Low (1): distress/hopelessness, no explicit self-harm
LOW_DISTRESS = re.compile(
    r"\b(hopeless|hopelessness|worthless|no\s+point|empty\s+inside|"
    r"can'?t\s+go\s+on|give\s+up|despair|overwhelmed|"
    r"don'?t\s+want\s+to\s+be\s+here|don'?t\s+want\s+to\s+wake\s+up|"
    r"wish\s+I\s+(wasn'?t|weren'?t)\s+(born|here)|"
    r"life\s+is\s+meaningless|nothing\s+left\s+to\s+live\s+for)\b",
    re.I,
)

# Exclude false positives
FALSE_POSITIVE = re.compile(
    r"at\s+the\s+risk\s+of|risk\s+of\s+(falling|sounding)",
    re.I,
)


def annotate_risk_level(text: str) -> int:
    """Return risk_level 0/1/2/3 per Zirikly-style rules."""
    if not isinstance(text, str) or not text.strip():
        return 0
    t = text.strip()

    if FALSE_POSITIVE.search(t):
        pass  # do not upgrade solely for "at the risk of" etc.

    if SEVERE_PLAN.search(t) or SEVERE_MEANS.search(t) or SEVERE_INTENT.search(t):
        if MODERATE_IDEATION.search(t) or MODERATE_SELFHARM.search(t):
            return 3

    if MODERATE_IDEATION.search(t) or MODERATE_SELFHARM.search(t):
        return 2

    if LOW_DISTRESS.search(t):
        return 1

    return 0

In [12]:
print("\n" + "=" * 60)
print("6. Rule-based pre-annotation (Zirikly-style)...")
print("=" * 60)

text_col = "text" if "text" in sampled.columns else "body"
if text_col not in sampled.columns and "title" in sampled.columns:
    sampled["text"] = (sampled["title"].fillna("") + " " + sampled.get("body", "").fillna("")).str.strip()
    text_col = "text"

sampled["risk_level"] = sampled[text_col].fillna("").apply(annotate_risk_level)

print("\nPre-annotation distribution:")
for lev, cnt in sampled["risk_level"].value_counts().sort_index().items():
    names = {0: "No Risk", 1: "Low", 2: "Moderate", 3: "Severe"}
    print(f"  {lev} {names.get(lev, lev)}: {cnt}")

out_path = DATA_DIR / "annotated_subset.csv"
sampled.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print("Recommend: human review of high/moderate-risk samples before running risk detection.")


6. Rule-based pre-annotation (Zirikly-style)...

Pre-annotation distribution:
  0 No Risk: 807
  1 Low: 52
  2 Moderate: 286
  3 Severe: 55

Saved: data/annotated_subset.csv
Recommend: human review of high/moderate-risk samples before running risk detection.


## 9. Summary

In [13]:
print("\n" + "=" * 60)
print("Done!")
print("=" * 60)
print(f"  - Processed corpus: data/processed_corpus.csv")
print(f"  - Sample for annotation: data/sample_for_annotation.csv")
print(f"  - Pre-annotated subset: data/annotated_subset.csv")
print("\nNext steps:")
print("  1. Run 02_topic_modeling.ipynb for BERTopic analysis")
print("  2. Run pbanjara/model_training.ipynb for risk classification")


Done!
  - Processed corpus: data/processed_corpus.csv
  - Sample for annotation: data/sample_for_annotation.csv
  - Pre-annotated subset: data/annotated_subset.csv

Next steps:
  1. Run 02_topic_modeling.ipynb for BERTopic analysis
  2. Run pbanjara/model_training.ipynb for risk classification
